# DPO FINE-TUNING: B2_SFT → B2_DPO
**Pipeline:**  
`Load B2 checkpoint` → `Tạo 100 preference pairs (chosen/rejected)` → `Train DPO (TRL)` → `So sánh metric SFT vs DPO`


In [ ]:
# !pip install -q transformers peft bitsandbytes accelerate trl \
#              evaluate bert_score rouge-score nltk openai huggingface_hub

In [ ]:
import os, json, random, re, warnings, gc, torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer as rs
import nltk, warnings
warnings.filterwarnings("ignore")
for p in ["wordnet","omw-1.4"]:
    try: nltk.data.find(f"corpora/{p}")
    except: nltk.download(p, quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download
import shutil

os.makedirs("data/openvivqa/dev", exist_ok=True)

def maybe_dl(filename, dest):
    p = hf_hub_download("uitnlp/OpenViVQA-dataset", filename, repo_type="dataset")
    if filename.endswith(".zip"):
        out = os.path.join(dest, filename.replace(".zip",""))
        if not os.path.exists(out):
            with zipfile.ZipFile(p) as z: z.extractall(dest)
    else:
        dst = os.path.join(dest, filename)
        if not os.path.exists(dst): shutil.copy(p, dst)

maybe_dl("dev-images.zip", "data/openvivqa/dev")
maybe_dl("vlsp2023_dev_data.json", "data/openvivqa")

with open("data/openvivqa/vlsp2023_dev_data.json", encoding="utf-8") as f:
    raw = json.load(f)

all_samples = []
for ann in raw["annotations"].values():
    fname = raw["images"].get(str(ann["image_id"]))
    if fname:
        p = f"data/openvivqa/dev/dev-images/{fname}"
        if os.path.exists(p):
            all_samples.append({"image": p, "question": ann["question"], "answer": ann["answer"]})

random.seed(42)
pool = random.sample(all_samples, min(150, len(all_samples)))
print(f"Pool: {len(pool)} mẫu — sẽ dùng 100 cặp đầu tiên để tạo preference data")

In [ ]:
BASE_ID     = "Qwen/Qwen2-VL-2B-Instruct"
SFT_ADAPTER = "./qwen2vl_finetuned_final"   # checkpoint B2

processor = AutoProcessor.from_pretrained(BASE_ID, max_pixels=313600)

print("Đang tải base model...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_ID,
    device_map={"": device},
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
)

print("Đang gắn SFT adapter (B2)...")
model_b2_sft = PeftModel.from_pretrained(base_model, SFT_ADAPTER)
model_b2_sft.eval()
print("B2 SFT sẵn sàng!")

In [ ]:
# B2 sau SFT vẫn có thể sinh ra câu trả lời sai hoặc lan man
# → dùng làm "rejected" trong DPO

@torch.no_grad()
def infer_qwen(model, sample, max_new_tokens=30):
    try:
        image = Image.open(sample["image"]).convert("RGB")
        msgs = [{"role":"user","content":[
            {"type":"image"},{"type":"text","text":sample["question"]}
        ]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp  = processor(text=[text], images=[image], padding=True, return_tensors="pt")
        inp  = {k: v.to(model.device) for k,v in inp.items()}
        out  = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
        trimmed = out[:, inp["input_ids"].shape[1]:]
        return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    except Exception as e:
        return ""

print("Đang lấy output từ B2_SFT (rejected answers)...")
rejected_list = []
for i, s in enumerate(pool[:100]):
    ans = infer_qwen(model_b2_sft, s)
    rejected_list.append(ans)
    if (i+1) % 10 == 0:
        print(f"  {i+1}/100 done | câu hỏi: '{s['question']}' | B2 nói: '{ans}'")

print(f"\nLấy xong {len(rejected_list)} câu trả lời rejected!")

In [ ]:
# PHƯƠNG ÁN A: Dùng Ground-Truth từ dataset (đơn giản, đáng tin cậy)
# PHƯƠNG ÁN B: Dùng GPT-4o rewrite (chất lượng cao hơn, cần API key)
# Mặc định dùng Phương án A. Bỏ comment phần B nếu bạn có OpenAI API key.
from openai import OpenAI
import base64

USE_GPT4O = False  # Đổi thành True nếu muốn dùng GPT-4o
OPENAI_API_KEY = ""
chosen_list = []

if USE_GPT4O and OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    def get_gpt4o_answer(sample):
        with open(sample["image"], "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role":"user","content":[
                {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img_b64}"}},
                {"type":"text","text":f"Trả lời thật ngắn gọn bằng tiếng Việt: {sample['question']}"}
            ]}],
            max_tokens=30
        )
        return resp.choices[0].message.content.strip()
    
    print("Đang dùng GPT-4o generate chosen answers...")
    for i, s in enumerate(pool[:100]):
        try:
            ans = get_gpt4o_answer(s)
        except:
            ans = s["answer"]  # fallback về GT nếu API lỗi
        chosen_list.append(ans)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/100 done")
else:
    # Phương án A: Ground-Truth từ dataset = câu trả lời chuẩn 100%
    print("Dùng Ground-Truth làm chosen answers (phương án mặc định)...")
    for s in pool[:100]:
        chosen_list.append(s["answer"])

print(f"\nCó {len(chosen_list)} chosen answers sẵn sàng!")
print(f"Ví dụ pair 1:")
print(f"  Câu hỏi : {pool[0]['question']}")
print(f"  Chosen  : {chosen_list[0]}")
print(f"  Rejected: {rejected_list[0]}")

In [ ]:
# Chỉ giữ lại những cặp mà chosen ≠ rejected (mới có ý nghĩa học)

def make_prompt(sample):
    msgs = [{"role":"user","content":[
        {"type":"image"},{"type":"text","text":sample["question"]}
    ]}]
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

valid_prompts, valid_chosen, valid_rejected, valid_img_paths = [], [], [], []
skipped = 0
for i, (s, ch, rej) in enumerate(zip(pool[:100], chosen_list, rejected_list)):
    # Bỏ qua nếu B2 trả lời y chang ground-truth (không cần train thêm)
    if ch.strip().lower() == rej.strip().lower():
        skipped += 1
        continue
    # Bỏ qua nếu rejected rỗng (lỗi inference)
    if not rej.strip():
        skipped += 1
        continue
    valid_prompts.append(make_prompt(s))
    valid_chosen.append(ch)
    valid_rejected.append(rej)
    valid_img_paths.append(s["image"])

print(f"Tổng cặp hợp lệ: {len(valid_prompts)} (bỏ qua {skipped} cặp trùng/rỗng)")
print(f"\n5 CẶP VÍ DỤ:")
for i in range(min(5, len(valid_prompts))):
    q = pool[i]["question"]
    print(f"  [{i+1}] Q: {q}")
    print(f"       ✓ Chosen  : {valid_chosen[i]}")
    print(f"       ✗ Rejected: {valid_rejected[i]}")
    print()

# Tạo HuggingFace Dataset
dpo_dataset = Dataset.from_dict({
    "prompt":   valid_prompts,
    "chosen":   valid_chosen,
    "rejected": valid_rejected,
    "img_path": valid_img_paths
})
print(f"Dataset DPO: {len(dpo_dataset)} cặp preference")

In [ ]:
# Giải phóng RAM trước khi train
del base_model, model_b2_sft
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

DPO_OUTPUT = "./qwen2vl_dpo_output"
os.makedirs(DPO_OUTPUT, exist_ok=True)

# Load lại model dưới dạng 4-bit để train DPO
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
print("Đang tải model cho DPO training...")
model_for_dpo = Qwen2VLForConditionalGeneration.from_pretrained(
    SFT_ADAPTER,         # Bắt đầu từ B2_SFT (không phải base model)
    quantization_config=bnb_cfg,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_for_dpo = prepare_model_for_kbit_training(model_for_dpo)
model_for_dpo.gradient_checkpointing_enable()
model_for_dpo.config.use_cache = False

lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
    lora_dropout=0.05
)
model_for_dpo = get_peft_model(model_for_dpo, lora_cfg)
model_for_dpo.print_trainable_parameters()

# Cấu hình DPO
dpo_config = DPOConfig(
    output_dir=DPO_OUTPUT,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,        # LR nhỏ hơn SFT rất nhiều
    beta=0.1,                  # Hệ số kiểm soát sự phân kỳ với model gốc
    loss_type="sigmoid",       # DPO loss chuẩn
    fp16=True if device=="cuda" else False,
    logging_steps=1,
    log_level="infor",
    save_steps=50,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    max_prompt_length=512,
    max_length=600,
)

tokenizer = processor.tokenizer
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainer = DPOTrainer(
    model=model_for_dpo,
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

print("\nBắt đầu DPO training...")
print(f"  Số cặp preference: {len(dpo_dataset)}")
print(f"  Beta: {dpo_config.beta} | LR: {dpo_config.learning_rate}")
trainer.train()

# Lưu model DPO
DPO_FINAL = "./qwen2vl_dpo_final"
trainer.save_model(DPO_FINAL)
processor.save_pretrained(DPO_FINAL)
print(f"\nĐã lưu B2_DPO tại: {DPO_FINAL}")

In [ ]:
del model_for_dpo, trainer
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

EVAL_SAMPLES = 50  # Đánh giá trên 50 mẫu test
test_pool = [s for s in all_samples if s not in pool[:100]]
eval_set  = random.sample(test_pool, min(EVAL_SAMPLES, len(test_pool)))
print(f"Mẫu đánh giá: {len(eval_set)}")

# Tải lại B2_SFT
print("\nTải B2_SFT...")
base_sft = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map={"": device},
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
)
model_b2_sft_eval = PeftModel.from_pretrained(base_sft, SFT_ADAPTER)
model_b2_sft_eval.eval()

# Tải B2_DPO
print("Tải B2_DPO...")
base_dpo = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    device_map={"": device},
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
)
model_b2_dpo_eval = PeftModel.from_pretrained(base_dpo, DPO_FINAL)
model_b2_dpo_eval.eval()

print("Cả hai model sẵn sàng!")

In [ ]:
smooth  = SmoothingFunction().method4
rouge_s = rs.RougeScorer(["rougeL"], use_stemmer=False)
try:
    bertscore = evaluate.load("bertscore")
except: bertscore = None

def compute_metrics(preds, gts):
    acc, bleu, rougeL, met = [], [], [], []
    for p, g in zip(preds, gts):
        pc = p.strip().lower(); gc_ = g.strip().lower()
        acc.append(1.0 if pc == gc_ else 0.0)
        bleu.append(sentence_bleu([gc_.split()], pc.split(), smoothing_function=smooth))
        rougeL.append(rouge_s.score(gc_, pc)["rougeL"].fmeasure)
        try:    met.append(meteor_score([gc_.split()], pc.split()))
        except: met.append(0.0)
    res = {
        "Accuracy":  round(np.mean(acc)*100, 2),
        "BLEU":      round(np.mean(bleu)*100, 2),
        "ROUGE-L":   round(np.mean(rougeL)*100, 2),
        "METEOR":    round(np.mean(met)*100, 2),
    }
    if bertscore:
        bs = bertscore.compute(predictions=preds, references=gts, lang="vi",
                               model_type="bert-base-multilingual-cased")
        res["BERTScore-F1"] = round(np.mean(bs["f1"])*100, 2)
    else: res["BERTScore-F1"] = "N/A"
    return res

gts_eval = [s["answer"] for s in eval_set]

print("Đang chạy inference B2_SFT...")
preds_sft = [infer_qwen(model_b2_sft_eval, s) for s in eval_set]

print("Đang chạy inference B2_DPO...")
preds_dpo = [infer_qwen(model_b2_dpo_eval, s) for s in eval_set]

metrics_sft = compute_metrics(preds_sft, gts_eval)
metrics_dpo = compute_metrics(preds_dpo, gts_eval)

print("\n" + "="*65)
print(f"  {'METRIC':<18} {'B2_SFT':>12} {'B2_DPO':>12} {'Δ (DPO-SFT)':>14}")
print("="*65)
for k in metrics_sft:
    v_sft = metrics_sft[k]; v_dpo = metrics_dpo[k]
    if isinstance(v_sft, float) and isinstance(v_dpo, float):
        delta = v_dpo - v_sft
        sign  = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
        print(f"  {k:<18} {v_sft:>12.2f} {v_dpo:>12.2f} {sign}{abs(delta):>12.2f}")
    else:
        print(f"  {k:<18} {str(v_sft):>12} {str(v_dpo):>12}")
print("="*65)

In [ ]:
import matplotlib.pyplot as plt

print("\n========== SO SÁNH CHI TIẾT 10 MẪU ĐẦU ==========")
print(f"{'Q':<45} {'GT':<25} {'SFT':<25} {'DPO':<25}")
print("-"*120)
for i in range(min(10, len(eval_set))):
    s   = eval_set[i]
    q   = s["question"][:43]
    gt  = gts_eval[i][:23]
    sft = preds_sft[i][:23]
    dpo = preds_dpo[i][:23]
    print(f"{q:<45} {gt:<25} {sft:<25} {dpo:<25}")

# Vẽ biểu đồ
metric_keys  = [k for k in metrics_sft if isinstance(metrics_sft[k], float)]
sft_vals     = [metrics_sft[k] for k in metric_keys]
dpo_vals     = [metrics_dpo[k] for k in metric_keys]
x = np.arange(len(metric_keys)); w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, sft_vals, w, label="B2_SFT", color="#4C72B0", alpha=0.9)
b2 = ax.bar(x + w/2, dpo_vals, w, label="B2_DPO", color="#DD8452", alpha=0.9)
ax.bar_label(b1, fmt="%.1f", padding=3, fontsize=9)
ax.bar_label(b2, fmt="%.1f", padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metric_keys, fontsize=11)
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 110)
ax.set_title("So sánh B2_SFT vs B2_DPO trên OpenViVQA", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("sft_vs_dpo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu biểu đồ: sft_vs_dpo.png")